# 6. Features del Original, Conteos y Optuna — Predicting Airline Satisfaction

El mejor modelo hasta acá es el LightGBM de `5_lgbm_seeds.ipynb`: **CV AUC = 0,95922** (`lr=0.02`,
3 semillas). Salvo `num_leaves` y el learning rate, sus hiperparámetros son los que se eligieron a
mano en el notebook 4, y nunca se probaron features nuevas para los árboles. Este notebook ataca las
dos cosas en tres etapas encadenadas:

1. **Features nuevas** (sección 3), comparadas fold a fold contra la base con `lr=0.05`:
   - **Consulta al dataset original.** Sumar las 129.880 filas reales como entrenamiento no sirvió
     (las distribuciones no coinciden), pero el **41 % de las filas sintéticas tiene exactamente las
     mismas 13 notas de encuesta que algún pasajero del original**: el generador copia patrones. Para
     cada fila se busca en el original la tasa de satisfacción de su combinación, sin mezclar las
     distribuciones.
   - **Conteos de combinaciones** en train + test: cuántas veces el generador repitió cada patrón.
2. **Optuna** (sección 4) sobre el mejor set de features: búsqueda en un fold con `lr=0.1`.
3. **CV final** (sección 5) con los parámetros encontrados y `lr=0.02`, comparada fold a fold contra
   el notebook 5.

Todas las corridas usan los **mismos folds** que los notebooks 3 a 5, y `deterministic=True` para
que las diferencias entre configuraciones no se mezclen con el ruido del multithreading.

> **Tiempo de ejecución: ~60 minutos** en CPU: ~20 min la prueba de features, ~20 min Optuna y
> ~20 min la CV final.

**Métrica:** ROC AUC — **Dataset:** Playground Series S6E10

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import optuna
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import os, sys, time
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (15, 6)
plt.rcParams["figure.dpi"] = 100

SEED     = 42
N_FOLDS  = 5
N_TRIALS = 30

## 1. Datos

El dataset original ("Airline Passenger Satisfaction", Kaggle) se baja una sola vez a
`../data/original/` (gitignoreado).

In [ ]:
ORIG_DIR = "../data/original"
if not os.path.exists(os.path.join(ORIG_DIR, "train.csv")):
    os.makedirs(ORIG_DIR, exist_ok=True)
    !kaggle datasets download -d teejmahal20/airline-passenger-satisfaction -p {ORIG_DIR} --unzip
print(sorted(os.listdir(ORIG_DIR)))

In [ ]:
train = pd.read_csv("../data/train.csv")
test  = pd.read_csv("../data/test.csv")
orig  = pd.concat([pd.read_csv(f"{ORIG_DIR}/train.csv"), pd.read_csv(f"{ORIG_DIR}/test.csv")],
                  ignore_index=True)

TARGET = "satisfaction"

crudas_num = ["Age", "Flight Distance", "Departure Delay in Minutes", "Arrival Delay in Minutes"]
encuesta = ["Inflight wifi service", "Departure/Arrival time convenient", "Ease of Online booking",
            "Gate location", "Food and drink", "Online boarding", "Seat comfort",
            "Inflight entertainment", "On-board service", "Leg room service", "Baggage handling",
            "Checkin service", "Cleanliness"]
crudas_cat = ["Gender", "Customer Type", "Type of Travel", "Class"]
crudas     = crudas_num + encuesta + crudas_cat

y        = train[TARGET].astype(int).values
orig["y"] = (orig["satisfaction"] == "satisfied").astype(int)

skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(train[crudas], y))

# OOF de los modelos anteriores, para las comparaciones pareadas
oof_nb4  = np.load("../models/4_lightgbm_oof.npy")
oof_nb5  = np.load("../models/5_lgbm_seeds_oof.npy")
test_nb5 = np.load("../models/5_lgbm_seeds_test.npy")


def aucs_por_fold(pred):
    return np.array([roc_auc_score(y[va], pred[va]) for _, va in folds])


def pareada(a, b, nombre):
    d = a - b
    t, p = stats.ttest_rel(a, b)
    print(f"  {nombre:36s} {d.mean():+.5f}  (desvio {d.std(ddof=1):.5f}, "
          f"gana {(d > 0).sum()}/{N_FOLDS}, t = {t:6.2f}, p = {p:.4f})")


aucs_nb4 = aucs_por_fold(oof_nb4)
aucs_nb5 = aucs_por_fold(oof_nb5)

print(f"Train: {len(train):,} | Test: {len(test):,} | Original: {len(orig):,} "
      f"(tasa {orig['y'].mean():.4f})")
print(f"Notebook 4 (lr 0.05):             {aucs_nb4.mean():.5f}")
print(f"Notebook 5 (lr 0.02, 3 semillas): {aucs_nb5.mean():.5f}")

## 2. Features Nuevas

Tres claves de distinta granularidad, para que siempre haya alguna con soporte:

- **`cat_top4`**: las 4 categóricas + las 4 notas más fuertes. Cubre casi todo el sintético.
- **`enc13`**: las 13 notas de la encuesta.
- **`cat3_enc13`**: las 13 notas + las categóricas salvo `Gender` (que es ruido).

Para cada clave:

- **`orig_tasa_*`**: tasa de satisfacción de esa combinación en el original, suavizada hacia la tasa
  global con `m = 5` filas ficticias (una combinación vista una sola vez no debería valer 0 o 1).
  NaN cuando la combinación no existe en el original; LightGBM aprende hacia dónde mandarlos.
- **`orig_n_*`**: cuántas veces aparece en el original (0 si no aparece).
- **`cnt_*`**: cuántas veces aparece en train + test sintéticos.

Ninguna usa el target del sintético, así que se calculan una sola vez sobre todo el dataset sin
riesgo de leakage entre folds.

In [ ]:
CLAVES = {
    "cat_top4":   crudas_cat + ["Online boarding", "Inflight wifi service",
                                "Inflight entertainment", "Seat comfort"],
    "enc13":      encuesta,
    "cat3_enc13": ["Customer Type", "Type of Travel", "Class"] + encuesta,
}
M_SUAVIZADO = 5
prior       = orig["y"].mean()
sintetico   = pd.concat([train[crudas], test[crudas]], ignore_index=True)

feats_orig, feats_cnt = [], []
for nombre, clave in CLAVES.items():
    g = orig.groupby(clave)["y"].agg(["sum", "count"]).reset_index()
    g[f"orig_tasa_{nombre}"] = (g["sum"] + M_SUAVIZADO * prior) / (g["count"] + M_SUAVIZADO)
    g = g.rename(columns={"count": f"orig_n_{nombre}"}).drop(columns="sum")

    c = sintetico.groupby(clave).size().rename(f"cnt_{nombre}").reset_index()

    for df in (train, test):
        # merge con how="left" conserva el orden de las filas de la izquierda
        extra = df[clave].merge(g, on=clave, how="left").merge(c, on=clave, how="left")
        df[f"orig_tasa_{nombre}"] = extra[f"orig_tasa_{nombre}"].values
        df[f"orig_n_{nombre}"]    = extra[f"orig_n_{nombre}"].fillna(0).values
        df[f"cnt_{nombre}"]       = extra[f"cnt_{nombre}"].values

    feats_orig += [f"orig_tasa_{nombre}", f"orig_n_{nombre}"]
    feats_cnt  += [f"cnt_{nombre}"]

    cubierto = train[f"orig_n_{nombre}"] > 0
    auc_tasa = roc_auc_score(y[cubierto], train.loc[cubierto, f"orig_tasa_{nombre}"])
    print(f"  {nombre:11s} cubre {cubierto.mean():6.1%} del train | AUC de orig_tasa sola "
          f"sobre las cubiertas: {auc_tasa:.4f} | cnt mediano {train[f'cnt_{nombre}'].median():.0f}")

for col in crudas_cat:
    dtype = pd.CategoricalDtype(categories=sorted(set(train[col]) | set(test[col])), ordered=False)
    train[col] = train[col].astype(dtype)
    test[col]  = test[col].astype(dtype)

## 3. Prueba de Features

Cuatro configuraciones con los parámetros del notebook 4 (`lr=0.05`, `num_leaves=255`). La base
se vuelve a correr acá en lugar de usar el OOF del notebook 4, para que todas salgan del mismo
código con `deterministic=True`.

In [ ]:
BASE = dict(
    objective="binary",
    n_estimators=10000,
    learning_rate=0.05,
    num_leaves=255,
    min_child_samples=40,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    random_state=SEED,
    deterministic=True,
    force_row_wise=True,
    n_jobs=-1,
    verbose=-1,
)


def entrenar(params, feats, tr_idx, va_idx, paciencia=150):
    m = lgb.LGBMClassifier(**params)
    m.fit(train[feats].iloc[tr_idx], y[tr_idx],
          eval_set=[(train[feats].iloc[va_idx], y[va_idx])], eval_metric="auc",
          callbacks=[lgb.early_stopping(paciencia, verbose=False)])
    return m


def cv(params, feats, etiqueta, con_test=False, paciencia=150):
    oof, test_pred, iters, modelos = np.zeros(len(train)), np.zeros(len(test)), [], []
    t0 = time.time()
    for tr_idx, va_idx in folds:
        m = entrenar(params, feats, tr_idx, va_idx, paciencia)
        oof[va_idx] = m.predict_proba(train[feats].iloc[va_idx])[:, 1]
        if con_test:
            test_pred += m.predict_proba(test[feats])[:, 1] / N_FOLDS
        iters.append(m.best_iteration_)
        modelos.append(m)
    aucs = aucs_por_fold(oof)
    print(f"  {etiqueta:22s} AUC = {aucs.mean():.5f} +/- {aucs.std():.5f} | "
          f"arboles {np.mean(iters):5.0f} | {(time.time() - t0) / 60:4.1f} min", flush=True)
    return dict(oof=oof, test=test_pred, aucs=aucs, modelos=modelos, feats=feats)


configs = {
    "base":             crudas,
    "+ original":       crudas + feats_orig,
    "+ conteos":        crudas + feats_cnt,
    "+ original + cnt": crudas + feats_orig + feats_cnt,
}

print(f"{'='*72}")
print("PRUEBA DE FEATURES - LightGBM lr=0.05, 5 folds")
print(f"{'='*72}")
res = {nombre: cv(BASE, feats, nombre) for nombre, feats in configs.items()}
print(f"{'='*72}\n")

for nombre in list(configs)[1:]:
    pareada(res[nombre]["aucs"], res["base"]["aucs"], f"{nombre} vs base")

FEATS = res[max(res, key=lambda k: res[k]["aucs"].mean())]["feats"]
GANADOR = [k for k in configs if configs[k] == FEATS][0]
print(f"\nSet de features elegido: {GANADOR} ({len(FEATS)} features)")

In [ ]:
# Importancia de las features nuevas dentro del set ganador
if GANADOR != "base":
    imps = pd.DataFrame([m.booster_.feature_importance(importance_type="gain")
                         for m in res[GANADOR]["modelos"]], columns=FEATS)
    imps = imps.div(imps.sum(axis=1), axis=0).mean().sort_values(ascending=False)
    print("Importancia (gain normalizado) del set ganador, top 15:")
    print(imps.head(15).round(4).to_string())

## 4. Optuna

Búsqueda sobre el set de features elegido, **sólo en el fold 1** y con `lr=0.1` para que cada prueba
tarde ~30-60 s. La primera prueba son los parámetros del notebook 4, así que el resultado nunca puede
quedar por debajo de la base en ese fold.

Elegir con el fold 1 y después medir en los 5 folds introduce un sesgo chico a favor en ese fold. Por
eso la comparación final se mira también en los folds 2 a 5.

In [ ]:
tr1, va1 = folds[0]


def objetivo(trial):
    params = {
        **BASE,
        "learning_rate":     0.1,
        "num_leaves":        trial.suggest_int("num_leaves", 31, 511, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 300, log=True),
        "subsample":         trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-3, 30, log=True),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "min_split_gain":    trial.suggest_float("min_split_gain", 0.0, 0.5),
        "max_bin":           trial.suggest_categorical("max_bin", [255, 511]),
    }
    m = entrenar(params, FEATS, tr1, va1, paciencia=100)
    trial.set_user_attr("arboles", m.best_iteration_)
    return roc_auc_score(y[va1], m.predict_proba(train[FEATS].iloc[va1])[:, 1])


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.enqueue_trial({"num_leaves": 255, "min_child_samples": 40, "subsample": 0.8,
                     "colsample_bytree": 0.7, "reg_lambda": 1.0, "reg_alpha": 1e-3,
                     "min_split_gain": 0.0, "max_bin": 255})

t0 = time.time()
study.optimize(objetivo, n_trials=N_TRIALS,
               callbacks=[lambda s, t: print(f"  prueba {t.number:2d}: AUC {t.value:.5f}  "
                                             f"(mejor {s.best_value:.5f}, "
                                             f"{(time.time() - t0) / 60:.1f} min)", flush=True)])

print(f"\nBase (prueba 0): {study.trials[0].value:.5f}  |  mejor: {study.best_value:.5f} "
      f"({study.best_value - study.trials[0].value:+.5f})")
print("Mejores parametros:")
for k, v in study.best_params.items():
    print(f"  {k:18s} {v}")

In [ ]:
hist = study.trials_dataframe()
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
axes[0].plot(hist["number"], hist["value"], "o", color="steelblue", alpha=0.6, label="prueba")
axes[0].plot(hist["number"], hist["value"].cummax(), color="tomato", lw=2, label="mejor hasta ahi")
axes[0].set_xlabel("Prueba")
axes[0].set_ylabel("AUC fold 1 (lr 0.1)")
axes[0].set_title("Historia de la busqueda")
axes[0].legend()
imp = optuna.importance.get_param_importances(study)
axes[1].barh(list(imp)[::-1], list(imp.values())[::-1], color="steelblue", edgecolor="black")
axes[1].set_title("Importancia de cada hiperparametro (fANOVA)")
plt.tight_layout()
plt.show()

## 5. CV Final

Parámetros de Optuna con `lr=0.02` y una sola semilla. Se compara contra el notebook 5, que promedia
3 semillas: si esta versión le gana igual, sumarle semillas después sólo puede ayudar.

> Esta es la otra celda pesada: ~20 minutos.

In [ ]:
FINAL = {**BASE, **study.best_params, "learning_rate": 0.02}

print(f"{'='*72}")
print("CV FINAL - LightGBM lr=0.02 con parametros de Optuna")
print(f"{'='*72}")
final = cv(FINAL, FEATS, f"{GANADOR} + optuna", con_test=True, paciencia=300)
print(f"{'='*72}\n")

print(pd.DataFrame({"nb 4 (lr 0.05)": aucs_nb4, "nb 5 (lr 0.02, 3 sem.)": aucs_nb5,
                    "base de la seccion 3": res["base"]["aucs"], "final": final["aucs"]},
                   index=[f"fold {i}" for i in range(1, N_FOLDS + 1)]).round(5).to_string())
print()
pareada(final["aucs"], aucs_nb5, "Final vs notebook 5")
pareada(final["aucs"][1:], aucs_nb5[1:], "Final vs notebook 5 (folds 2-5)")
pareada(final["aucs"], aucs_nb4, "Final vs notebook 4")

## 6. Mezcla con el Notebook 5 y Submission

Promedio de rankings con el notebook 5, con el peso elegido sobre el OOF. Se envía la mejor de las
dos opciones por CV.

In [ ]:
r_fin = pd.Series(final["oof"]).rank(pct=True).values
r_nb5 = pd.Series(oof_nb5).rank(pct=True).values
mezcla = {w: roc_auc_score(y, (1 - w) * r_fin + w * r_nb5) for w in np.round(np.arange(0, 0.91, 0.1), 1)}
for w, a in mezcla.items():
    print(f"  peso notebook 5 {w:.1f}: AUC OOF = {a:.5f}")

W_NB5    = max(mezcla, key=mezcla.get)
aucs_mix = aucs_por_fold((1 - W_NB5) * r_fin + W_NB5 * r_nb5)
print(f"\nMejor peso del notebook 5: {W_NB5}")
pareada(aucs_mix, final["aucs"], "Mezcla vs final")

mix_test = ((1 - W_NB5) * pd.Series(final["test"]).rank(pct=True).values
            + W_NB5 * pd.Series(test_nb5).rank(pct=True).values)

np.save("../models/6_lgbm_optuna_oof.npy",  final["oof"])
np.save("../models/6_lgbm_optuna_test.npy", final["test"])

subs = {
    "final":  ("../submissions/6_lgbm_optuna_submission.csv", final["test"], final["aucs"].mean()),
    "mezcla": ("../submissions/6_lgbm_optuna_nb5_blend_submission.csv", mix_test, aucs_mix.mean()),
}
for nombre, (ruta, pred, auc) in subs.items():
    pd.DataFrame({"id": test["id"], TARGET: pred}).to_csv(ruta, index=False)
    print(f"{nombre:7s} CV AUC = {auc:.5f}  ->  {ruta}")

ENVIO = "mezcla" if W_NB5 > 0 and aucs_mix.mean() > final["aucs"].mean() else "final"
FILE, _, CV_ENVIO = subs[ENVIO]
print(f"\nSe envia: {ENVIO}")
print(f"Probabilidad media predicha: {final['test'].mean():.4f}  (tasa base: {y.mean():.4f})")

In [ ]:
# Chequeo de sanidad contra sample_submission (formato e ids)
sample = pd.read_csv("../data/sample_submission.csv")
for ruta, _, _ in subs.values():
    sub = pd.read_csv(ruta)
    assert list(sub.columns) == list(sample.columns), "columnas no coinciden"
    assert len(sub) == len(sample), "cantidad de filas no coincide"
    assert (sub["id"].values == sample["id"].values).all(), "los ids no estan alineados"
    assert sub[TARGET].between(0, 1).all(), "hay probabilidades fuera de [0, 1]"
    assert sub[TARGET].notna().all(), "hay nulos en la prediccion"
print("Chequeos OK - submissions listas para enviar.")

## 7. Envío a Kaggle vía API

In [ ]:
COMPETITION = "playground-series-s6e10"
MSG = (f"LightGBM {GANADOR} + optuna lr=0.02"
       + (f" + nb5 rank blend w={W_NB5}" if ENVIO == "mezcla" else "")
       + f" - CV AUC={CV_ENVIO:.5f}")

!kaggle competitions submit -c {COMPETITION} -f "{FILE}" -m "{MSG}"

In [ ]:
# Score publico de los envios (esperar unos segundos a que Kaggle lo puntue)
!kaggle competitions submissions -c {COMPETITION}

## 8. Conclusiones

_(se completan después de ejecutar)_